# ការកក់សណ្ឋាគារជាមួយ Middleware សមាជិកអាទិភាព

កំណត់ត្រានេះបង្ហាញពី **middleware ផ្អែកលើមុខងារ** ដោយប្រើ Microsoft Agent Framework។ យើងបានបន្ថែមខ្នាត middleware មួយដែលផ្តល់អាទិភាពពិសេសសម្រាប់សមាជិកដែលមានអាទិភាព នៅលើករណីនៃសូម្បីតែ workflow មានលក្ខខណ្ឌ។

## អ្វីដែលអ្នកនឹងរៀន:
1. **Middleware ផ្អែកលើមុខងារ**: ឈរកាត់ និងផ្លាស់ប្តូរវិលលទ្ធផលមុខងារ
2. **ចូលដល់ Context**: អាន និងផ្លាស់ប្តូរ `context.result` បន្ទាប់ពីការអនុវត្ត
3. **អនុវត្តត្រឹមត្រង់អាជីវកម្ម**: អត្ថប្រយោជន៍សម្រាប់សមាជិកអាទិភាព
4. **ការប្រែប្រួលលទ្ធផល**: ផ្លាស់ប្តូរវិលលទ្ធផលមុខងារតាមស្ថានភាពអ្នកប្រើ
5. **Workflow ដដែល ប៉ុន្តែមានលទ្ធផលខុសគ្នា**: ការផ្លាស់ប្តូរយើងអាចបង្កើតដោយ middleware

## រចនាសម្ព័ន្ធ Workflow ជាមួយ Middleware៖

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## ភាពខុសគ្នាសំខាន់ពី Workflow មានលក្ខខណ្ឌ៖

**គ្មាន Middleware** (14-conditional-workflow.ipynb):
- ប៉ារីសគ្មានបន្ទប់ → ប្តូរសំរាប់ alternative_agent

**មាន Middleware** (កំណត់ត្រានេះ):
- អ្នកប្រើប្រាស់ទៀងទាត់ + ប៉ារីស → គ្មានបន្ទប់ → ប្តូរសំរាប់ alternative_agent
- សមាជិកអាទិភាព + ប៉ារីស → 🌟 Middleware បញ្ជាក់អាទិភាព! → មានបន្ទប់ → ប្តូរសំរាប់ booking_agent

## ពាក្យបញ្ជាក់ចាំបាច់៖
- មាន Microsoft Agent Framework តម្លើងរួច
- យល់ដឹងអំពី workflow មានលក្ខខណ្ឌ (មើល 14-conditional-workflow.ipynb)
- យកសញ្ញា GitHub ឬ key API OpenAI
- មានចំណេះដឹងមូលដ្ឋានអំពីគំរូ middleware


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## ជំហានទី 1៖ កំណត់ ម៉ូដែល Pydantic សម្រាប់លទ្ធផលដែលមានរចនាសម្ព័ន្ធ

ម៉ូដែលទាំងនេះកំណត់ **ស្គីម៉ា** ដែលភ្នាក់ងារនឹងត្រឡប់មកវិញ។ យើងបានបន្ថែមវាល `priority_override` ដើម្បីតាមដានពេល middleware បម្លែងលទ្ធផលអាចប្រើបាន។


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## ជំហានទី 2៖ កំណត់មូលដ្ឋានទិន្នន័យសមាជិកអាទិភាព

សម្រាប់ការបង្ហាញនេះ យើងនឹងច្បាស់លាស់មូលដ្ឋានទិន្នន័យសមាជិកអាទិភាព។ ក្នុងការផលិត វានឹងស្វែងរកក្នុងមូលដ្ឋានទិន្នន័យពិត ឬ API។

**សមាជិកអាទិភាព៖**
- `alice@example.com` - សមាជិក VIP
- `bob@example.com` - សមាជិក Premium  
- `priority_user` - គណនីប្រើប្រាស់សាកល្បង


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## ជំហានទី 3: បង្កើតឧបករណ៍កក់សណ្ឋាគារ

ដូចជាលំនាំនៃការធ្វើការជម្រុញតែក្នុងពេលនេះ វានឹងត្រូវបានរារាំងដោយ middleware!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## ជំហ៊ាន ៤៖ 🌟 បង្កើត Middleware សម្រាប់ពិនិត្យអាទិភាព (មុខងារសំខាន់!)

នេះគឺជា **មុខងារស្នូល** នៃកំណត់ត្រានេះ។ Middleware មានមុខងារ៖

១. **ឆក់យក** ការហៅមុខងារ hotel_booking
២. **អនុវត្ត** មុខងារដដែលដោយហៅ `next(context)`
៣. **ពិនិត្យ** លទ្ធផលនៅក្នុង `context.result`
៤. **ប្តូរ** លទ្ធផល ប្រសិនបើអ្នកប្រើប្រាស់ជាអាទិភាព និងមិនមានបន្ទប់ទេ
៥. **បញ្ជូន** លទ្ធផលបានកែប្រែនៅត្រឡប់ទៅភ្នាក់ងារ

**លំនាំសំខាន់៖**
```python
async def my_middleware(context, next):
    await next(context)  # ប្រតិបត្តិការឧបករណ៍
    # ឥឡូវ context.result មានលទ្ធផលនៃអនុគមន៍
    if some_condition:
        context.result = new_value  # ជំនួស!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## ជំហាន​ទី 5: កំណត់​មុខងារ​លក្ខខណ្ឌ​សម្រាប់​ការបញ្ជាទិញ

មុខងារ​លក្ខខណ្ឌ​ដូច​ជា​ការច្រក​ផ្លូវ​ធម្មតា - ពួកវាស៊ើបអង្កេតលទ្ធផលដែលមានរចនាសម្ព័ន្ធ ដើម្បីកំណត់ការបញ្ជាទិញ។


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## ជំហានទី ៦: បង្កើតកម្មវិធីប្រតិបត្តិការបង្ហាញផ្ទាល់ខ្លួន

កម្មវិធីប្រតិបត្តិការដូចគ្នានឹងមុន - បង្ហាញលទ្ធផលសកម្មភាពចុងក្រោយ។


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## ជំហានទី 7៖ ផ្ទុកអថិជនបរិស្ថាន

កំណត់រចនាសម្ព័ន្ធអតិថិជន LLM (Microsoft Foundry / Azure OpenAI)។


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## ជំហានទី ៨: បង្កើតភ្នាក់ងារត្រូវបានគ្រប់គ្រងដោយ AI ជាមួយដំណាក់មធ្យម

**ភាពខុសគ្នាសំខាន់:** ពេលបង្កើត availability_agent យើងផ្តល់ប៉ារ៉ាម៉ែត្រ `middleware` ទៅ!

នេះជាវិធីដែលយើងបញ្ចូល priority_check_middleware ចូលទៅក្នុងបណ្តាញហៅមុខងាររបស់ភ្នាក់ងារ។


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## ជំហានទី 9: បង្កើតសំណុំការងារ

រចនាសម្ព័ន្ធសំណុំការងារដូចម្ដេចដដែលដូចមុន - ផ្លូវចំនួនដោយផ្អែកលើការចំណាស់អាចប្រើបាន។


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## ជំហានទី ១០៖ ករណីសាកល្បង ១ - អ្នកប្រើប្រាស់ធម្មតានៅទីក្រុងប៉ារីស (មិនមានការបដិសេធ)

អ្នកប្រើប្រាស់ធម្មតាដែលព្យាយាមកក់បន្ទប់នៅប៉ារីស → គ្មានបន្ទប់ → ផ្ញើទៅ alternative_agent


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## ជំំហៀងទី ១១៖ ករណីតេស្តទី ២ - 🌟 អ្នកប្រើប្រាស់អាទិភាពនៅទីក្រុងប៉ារីស (ជាមួយការជំនួស!)

សមាជិកអាទិភាពព្យាយាមកក់ប៉ារីស → មិនមានបន្ទប់នៅដំបូង → 🌟 មេឌៀវែរ ដកឃ្លាំង! → ផ្ញើទៅ booking_agent

**នេះជាការបង្ហាញសំខាន់នៃអំណាចមេឌៀវែរ!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## ជំហាន ១២៖ ករណីសាកល្បងទី ៣ - អ្នកប្រើប្រាស់អាទិភាពនៅស្ទុកហ្កែល (មានរួចហើយ)

អ្នកប្រើប្រាស់អាទិភាពព្យាយាមស្ទុកហ្កែល → មានបន្ទប់ → មិនចាំបាច់បដិសេធ → បញ្ជូនទៅ booking_agent

នេះបង្ហាញថា middleware ប្រតិបត្តិការតែមួយនៅពេលត្រូវការ​ប៉ុណ្ណោះ!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## ចំណុចសំខាន់ និង យល់ដឹងអំពី Middleware

### ✅ អ្វីដែលអ្នកបានរៀន៖

#### **1. គំរូ Middleware ដោយផ្អែកលើ Function**

Middleware ចាប់ការហៅ function ដោយប្រើ async function ងាយស្រួល៖

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # មុនការប្រតិបត្តិការកម្មវិធី
    print("Intercepting...")
    
    # ប្រតិបត្តិការកម្មវិធី
    await next(context)
    
    # បន្ទាប់ពីការប្រតិបត្តិការកម្មវិធី - ពិនិត្យលទ្ធផល
    if context.result:
        # ផ្លាស់ប្តូរលទ្ធផលប្រសិនបើត្រូវការ
        context.result = modified_value
```

#### **2. ការចូលដល់ Context និង ការប្តូរប្រសិទ្ធផល**

- `context.function` - ចូលដល់ function ដែលត្រូវបានហៅ
- `context.arguments` - អាន arguments នៃ function
- `context.kwargs` - ចូលដល់ parameters បន្ថែមផ្សេងៗ
- `await next(context)` - ប្រតិបត្តិ function
- `context.result` - អាន/កែប្រែចេញពី function

#### **3. ការអនុវត្តតាមតុល្យភាពជំនួញ**

Middleware របស់យើងអនុវត្តបំណងអោយមានអាទិភាពសមាជិក៖
- **អ្នកប្រើទូទៅ**: មិនបម្លែង អនុវត្តផ្លូវធម្មតា
- **អ្នកប្រើអាទិភាព**: ប្ដូរ "មិនមានទំនុកចិត្ត" → "អាចប្រើបាន"
- **តុល្យភាពលក្ខខណ្ឌ**: ប្ដូរតែមួយពេលដែលចាំបាច់

#### **4. workflow ដូចគ្នា ប៉ុន្តែផលវិបាកខុសគ្នា**

ខ្លាំងរបស់ middleware គឺ៖
- ✅ មិនមានការផ្លាស់ប្តូរទៅលើរចនាសម្ព័ន្ធ workflow
- ✅ មិនបានបម្លែង function ឧបករណ៍
- ✅ មិនបានប្ដូរតុល្យភាព routing logic
- ✅ របៀបធ្វើមានតែលើ middleware → ប្លែកការប្រតិបត្តិ!

### 🚀 ការអនុវត្តនៅក្នុងពិភពដ៏ពិត:

1. **លក្ខណៈពិសេស VIP/Premium**
   - ប្ដូរគន្លងកំណត់ចំនួនសម្រាប់អ្នកប្រើ premium
   - ផ្តល់ការចូលដំណើរការអាទិភាពទៅធនធាន
   - បើកលក្ខណៈពិសេស premium ដោយ δυναμικά

2. **A/B Testing**
   - ផ្លូវអោយអ្នកប្រើទៅអនុវត្តខុសគ្នា
   - សាកល្បងលក្ខណៈពិសេសជាមួយអ្នកប្រើជាក់លាក់
   - ចេញលក្ខណៈពិសេសតាមជំហាន

3. **សុវត្ថិភាព និង ប្រាក់ក្នុតបញ្ជាក់**
   - គ្រប់គ្រងហៅ function
   - បិទប្រតិបត្តិការដែលមានភាពម៉ាស
   - អនុវត្តតុល្យភាពជំនួញ

4. **សម្រួលការសម្រួលប្រសិទ្ធភាព**
   - បង្ហុកលទ្ធផលសម្រាប់អ្នកប្រើជាក់លាក់
   - មិនអនុវត្តន៍ជំនួញថ្លៃក្នុងពេលត្រឹមត្រូវ
   - កំណត់ធនធានឲ្យមានការផ្លាស់ប្តូរទៅជាប្រតិបត្តិការ

5. **ដោះស្រាយកំហុស និង ចម្លងបដិសណ្ឋារកិច្ច**
   - ចាប់កំហុស និង ដោះស្រាយដោយស្លូតបូត
   - អនុវត្តតុល្យភាព retry
   - ត្រឡប់ទៅអនុវត្តន៍ជម្រើសផ្សេងទៀត

6. **កំណត់ហេតុ និង ត្រួតពិនិត្យ**
   - តាមដានពេលវេលាប្រតិបត្តិ function
   - កត់ត្រាពិន្ទុ និងលទ្ធផល
   - ត្រួតពិនិត្យលំនាំប្រើប្រាស់

### 🔑 ភាពខុសគ្នាសំខាន់ពី Decorators:

| លក្ខណៈ | Decorator | Middleware |
|---------|-----------|------------|
| **វិសាលភាព** | មួយ function តែប៉ុណ្ណោះ | មានទាំងអស់នៅក្នុង agent |
| **ភាពបត់បែន** | កំណត់ជាមុននៅពេលការពិធីរៀបចំ | ប្ដូរតាមពេលរត់កម្មវិធី |
| **Context** | មានដែនកំណត់ | មាន context agent ពេញលេញ |
| **រចនាសម្ព័ន្ធ** | decorators តែមួយចំនួន | រង្វាស់ middleware |
| **មានជំនាញ agent** | មិនមាន | មាន (ចូលដល់ស្ថានភាព agent) |

### 📚 ពេលវេលាដែលត្រូវប្រើ Middleware:

✅ **ប្រើ middleware នៅពេល៖**
- អ្នកត្រូវកែប្រែការប្រព្រឹត្តទៅលើ user/session state
- អ្នកចង់អនុវត្តតុល្យភាពទៅច្រើន function
- អ្នកត្រូវចូលដល់ context ជំរើស agent
- អ្នកកំពុងអនុវត្តចំណុចកាត់ឆ្នៃ( logging, auth, ល។)

❌ **កុំប្រើ middleware នៅពេល៖**
- ការត្រួតពិនិត្យបញ្ចូលសាមញ្ញ(ប្រើ Pydantic)
- តុល្យភាព function ជាក់លាក់ (រក្សាទុកនៅក្នុង function)
- ការផ្លាស់ប្តូរតែមួយដង (គ្រាន់តែប្ដូរ function)

### 🎓 គំរូកម្រិតខ្ពស់:

```python
# មធ្យមភាសារពីរឬច្រើន (លំដាប់អនុវត្តសំខាន់!)
middleware=[
    logging_middleware,      # កំណត់ហេតុនៅដំបូង
    auth_middleware,         # បន្ទាប់មកពិនិត្យអត្តសញ្ញាណ
    cache_middleware,        # បន្ទាប់មកពិនិត្យខ្ទង់បញ្ចី
    rate_limit_middleware,   # បន្ទាប់មកកំណត់កម្រាស់អត្រា
    priority_check_middleware  # ចុងក្រោយពិនិត្យអាទិភាព
]

# អនុវត្តមធ្យមភាសាតាមលក្ខខណ្ឌ
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # កែប្រែលទ្ធផល
    else:
        # ពន្លឿនអនុវត្តឱ្យបានស្រួល
        context.result = cached_value
```

### 🔗 គំនិតដែលពាក់ព័ន្ធ:

- **Middleware agent**: ចាប់ agent.run() អំពើហៅ។
- **Middleware function**: ចាប់ហៅ function នៃឧបករណ៍ (ដែលយើងប្រើ!)
- **Middleware Pipeline**: តំណភ្ជាប់ middleware ដែលដំណើរការតាមលំដាប់
- **បន្ត context**: ផ្ញើស្ថានភាពតាមរយៈច្រវ៉ាក់ middleware


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
